In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
import time
from collections import defaultdict

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
import hdbscan
from sklearn.metrics import accuracy_score

In [2]:
N_PER_CLASS = 10_000
RANDOM_STATE = 42
HDBSCAN_TRAIN_SIZE = 5_000

In [3]:
cat_path = Path("datos/full_numpy_bitmap_cat.npy")
dog_path = Path("datos/full_numpy_bitmap_dog.npy")

cats = np.load(cat_path)
dogs = np.load(dog_path)

print(f"Gatos disponibles: {len(cats):,}")
print(f"Perros disponibles: {len(dogs):,}")
print(f"Forma de cada dibujo: {cats.shape[1:]}")

Gatos disponibles: 123,202
Perros disponibles: 152,159
Forma de cada dibujo: (784,)


In [4]:
rng = np.random.default_rng(RANDOM_STATE)

cat_idx = rng.choice(len(cats), N_PER_CLASS, replace=False)
dog_idx = rng.choice(len(dogs), N_PER_CLASS, replace=False)

cats = cats[cat_idx]
dogs = dogs[dog_idx]

X = np.concatenate([cats, dogs], axis=0)

y = np.concatenate([
    np.zeros(N_PER_CLASS, dtype=np.uint8),
    np.ones(N_PER_CLASS, dtype=np.uint8)
])

In [5]:
rng = np.random.default_rng(42)

indices_sobrantes = rng.choice(
    len(X),
    size=10,
    replace=False
)

X_sobrante = X[indices_sobrantes]
y_sobrante = y[indices_sobrantes]

# Eliminar las imagenes escogidas
mascara = np.ones(len(X), dtype=bool)
mascara[indices_sobrantes] = False

X = X[mascara]
y = y[mascara]

In [6]:
shuffle_idx = rng.permutation(len(X))
X = X[shuffle_idx]
y = y[shuffle_idx]

In [7]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

In [8]:
X_train = X_train.astype(np.float32) / 255.0
X_val   = X_val.astype(np.float32) / 255.0
X_test  = X_test.astype(np.float32) / 255.0

In [9]:
output_path = Path("datos/gatos_perros_dataset.npz")

np.savez_compressed(
    output_path,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    X_test=X_test,
    y_test=y_test
)

print("\nDataset preparado:")
print(f"Train:      {X_train.shape[0]:,} dibujos")
print(f"Validation: {X_val.shape[0]:,} dibujos")
print(f"Test:       {X_test.shape[0]:,} dibujos")
print(f"Total:      {len(X):,} dibujos")
print(f"\nArchivo guardado en: {output_path}")


Dataset preparado:
Train:      13,993 dibujos
Validation: 2,998 dibujos
Test:       2,999 dibujos
Total:      19,990 dibujos

Archivo guardado en: datos\gatos_perros_dataset.npz


In [10]:
rng = np.random.default_rng(42)
hdb_idx = rng.choice(len(X_train), HDBSCAN_TRAIN_SIZE, replace=False)

X_hdbscan = X_train[hdb_idx]
y_hdbscan = y_train[hdb_idx]

In [11]:
def print_resultado(resultados:dict) -> None:
    print(f"\n{'=' * 60}")
    print(resultados["modelo"])
    print("=" * 60)

    print(f"Tiempo entrenamiento: {resultados["tiempo_entrenamiento_s"]:.3f} s")
    print(f"Tiempo predicción:    {resultados["tiempo_prediccion_s"]:.3f} s")
    print(f"Accuracy:             {resultados['accuracy']:.4f}")

In [12]:
def evaluar_imagenes_singulares(modelos, X_sobrante, nombres_modelos):
    resultados = []

    for modelo, nombre in zip(modelos, nombres_modelos):

        for i, imagen in enumerate(X_sobrante):

            imagen_input = imagen.reshape(1, -1)

            inicio = time.perf_counter()

            if nombre == "HDBSCAN":

                cluster, _ = hdbscan.approximate_predict(
                    modelo,
                    imagen_input
                )

                cluster = cluster[0]

                if cluster == -1:
                    etiqueta = -1

                elif cluster in modelo.cluster_a_clase_:
                    etiqueta = modelo.cluster_a_clase_[cluster]

                else:
                    etiqueta = -1

            else:
                etiqueta = modelo.predict(imagen_input)[0]

            tiempo_prediccion = time.perf_counter() - inicio

            resultados.append({
                "modelo": nombre,
                "indice_imagen": i,
                "imagen": imagen.copy(),
                "etiqueta": etiqueta,
                "tiempo_prediccion_s": tiempo_prediccion
            })

    return resultados

In [13]:
def evaluar_clasificador(nombre, modelo, X_sobrante=None):

    # Entrenamiento

    inicio = time.perf_counter()

    modelo.fit(X_train, y_train)

    tiempo_entrenamiento = time.perf_counter() - inicio

    # Prediccion

    inicio = time.perf_counter()

    pred = modelo.predict(X_test)

    tiempo_prediccion = time.perf_counter() - inicio

    # Resultados

    resultados = {
        "modelo": nombre,
        "tiempo_entrenamiento_s": tiempo_entrenamiento,
        "tiempo_prediccion_s": tiempo_prediccion,
        "accuracy": accuracy_score(y_test, pred)
    }

    print_resultado(resultados)

    # Experimento

    resultados_individuales = None

    if X_sobrante is not None:
        resultados_individuales = evaluar_imagenes_singulares(
            [modelo],
            X_sobrante,
            [nombre]
        )

    return modelo, pred, resultados, resultados_individuales

In [14]:
def entrenar_hdbscan(X_train, y_train, X_test, y_test,
                     X_sobrante=None):

    modelo = hdbscan.HDBSCAN(
        min_cluster_size=50,
        min_samples=10,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True
    )

    # -------------------------
    # Entrenamiento
    # -------------------------

    inicio = time.perf_counter()

    modelo.fit(X_train)

    tiempo_entrenamiento = time.perf_counter() - inicio

    # -------------------------
    # Crear mapa: cluster -> clase
    # -------------------------

    clusters = modelo.labels_

    mapa_clusters = defaultdict(list)

    for cluster, clase in zip(clusters, y_train):

        if cluster != -1:
            mapa_clusters[cluster].append(clase)

    cluster_a_clase = {}

    for cluster, clases in mapa_clusters.items():

        valores, cantidades = np.unique(
            clases,
            return_counts=True
        )

        clase_mayoritaria = valores[np.argmax(cantidades)]

        cluster_a_clase[cluster] = clase_mayoritaria

    # Guardamos el mapa dentro del modelo para utilizarlo
    # posteriormente en las predicciones individuales.

    modelo.cluster_a_clase_ = cluster_a_clase

    # -------------------------
    # Prediccion sobre TEST
    # -------------------------

    inicio = time.perf_counter()

    clusters_predichos, _ = hdbscan.approximate_predict(
        modelo,
        X_test
    )

    tiempo_prediccion = time.perf_counter() - inicio

    y_pred = []

    for cluster in clusters_predichos:

        if cluster == -1:
            y_pred.append(-1)

        elif cluster in cluster_a_clase:
            y_pred.append(cluster_a_clase[cluster])

        else:
            y_pred.append(-1)

    y_pred = np.array(y_pred)

    # -------------------------
    # Resultados globales
    # -------------------------

    resultados = {
        "modelo": "HDBSCAN",
        "accuracy": accuracy_score(y_test, y_pred),
        "tiempo_entrenamiento_s": tiempo_entrenamiento,
        "tiempo_prediccion_s": tiempo_prediccion
    }

    print_resultado(resultados)

    # -------------------------
    # Experimento individual
    # -------------------------

    resultados_individuales = None

    if X_sobrante is not None:
        resultados_individuales = evaluar_imagenes_singulares(
            [modelo],
            X_sobrante,
            ["HDBSCAN"]
        )

    return modelo, resultados, resultados_individuales

In [15]:
svm_model = SVC(
    kernel="rbf",
    C=10,
    gamma="scale",
    cache_size=2048
)

svm_model, svm_pred, svm_results, svm_individuales = evaluar_clasificador(
    "SVM",
    svm_model,
    X_sobrante
)


SVM
Tiempo entrenamiento: 305.532 s
Tiempo predicción:    11.735 s
Accuracy:             0.8393


In [16]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42
)

rf_model, rf_pred, rf_results, rf_individuales = evaluar_clasificador(
    "Random Forest",
    rf_model,
    X_sobrante
)


Random Forest
Tiempo entrenamiento: 2.618 s
Tiempo predicción:    0.048 s
Accuracy:             0.8203


In [17]:
mlp_model = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    batch_size=128,
    learning_rate_init=0.001,
    max_iter=30,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=5,
    random_state=42
)

mlp_model, mlp_pred, mlp_results, mlp_individuales = evaluar_clasificador(
    "Red neuronal MLP",
    mlp_model,
    X_sobrante
)


Red neuronal MLP
Tiempo entrenamiento: 2.595 s
Tiempo predicción:    0.006 s
Accuracy:             0.8393


In [18]:
hdbscan_model, hdbscan_results, hdbscan_individuales = entrenar_hdbscan(
    X_train,
    y_train,
    X_test,
    y_test,
    X_sobrante
)


HDBSCAN
Tiempo entrenamiento: 264.016 s
Tiempo predicción:    27.471 s
Accuracy:             0.2064


In [19]:
variables_resultados = [
    svm_results,
    rf_results,
    mlp_results,
    hdbscan_results
]

df_resultados_globales = pd.DataFrame(variables_resultados)

df_resultados_globales.to_csv(
    "resultados/resultados_globales.csv",
    index=False
)

In [20]:
resultados_individuales = (
    svm_individuales
    + rf_individuales
    + mlp_individuales
    + hdbscan_individuales
)

imagenes_individuales = np.array([
    resultado["imagen"]
    for resultado in resultados_individuales
])

df_resultados_individuales = pd.DataFrame([
    {
        "modelo": resultado["modelo"],
        "indice_imagen": resultado["indice_imagen"],
        "etiqueta": resultado["etiqueta"],
        "tiempo_prediccion_s": resultado["tiempo_prediccion_s"]
    }
    for resultado in resultados_individuales
])

np.savez_compressed(
    "resultados/experimento_individual_imagenes.npz",
    imagenes=imagenes_individuales
)

df_resultados_individuales.to_csv(
    "resultados/experimento_individual_resultados.csv",
    index=False
)